In [ ]:
import numpy as np

import optuna

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.model_selection import StratifiedKFold, cross_val_score

from stroke_risk.ingest.load_data import load_stroke_data

import warnings

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

print("Imports OK")

In [ ]:
X, y = load_stroke_data()
print(X.shape)
print(y.shape)

In [ ]:
# Column types
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = X.select_dtypes(include=np.number).columns.tolist()

print(f"Categorical : {cat_cols}")
print(f"Numerical   : {num_cols}")

# Quick check: no remaining nulls
print(f"Nulls in X    : {X.isnull().sum().sum()}")
print(f"Nulls in y : {y.isnull().sum().sum()}")

In [ ]:
# Percentage of missing values
round(X.isnull().sum() / X.shape[1], 2)

In [ ]:
num_pipeline_log = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

num_pipeline_tree = Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))])

In [ ]:
cat_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(sparse_output=False, handle_unknown="ignore")),
    ]
)

In [ ]:
preprocessor_log = ColumnTransformer(
    [("num", num_pipeline_log, num_cols), ("cat", cat_pipeline, cat_cols)]
)

preprocessor_tree = ColumnTransformer(
    [("num", num_pipeline_tree, num_cols), ("cat", cat_pipeline, cat_cols)]
)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

log_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor_log),
        (
            "model",
            LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced"),
        ),
    ]
)

rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor_log),
        (
            "model",
            RandomForestClassifier(
                max_depth=5, random_state=42, class_weight="balanced"
            ),
        ),
    ]
)

scale_val_xgb = len(y[y == 0]) / len(y[y == 1])

xgb_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor_log),
        (
            "model",
            XGBClassifier(max_depth=5, random_state=42, scale_pos_weight=scale_val_xgb),
        ),
    ]
)

In [ ]:
log_scores = cross_val_score(log_pipeline, X, y, cv=cv, n_jobs=-1, scoring="roc_auc")
rf_scores = cross_val_score(rf_pipeline, X, y, cv=cv, n_jobs=-1, scoring="roc_auc")
xgb_scores = cross_val_score(xgb_pipeline, X, y, cv=cv, n_jobs=-1, scoring="roc_auc")

print(
    f"Log-reg score mean: {log_scores.mean():.2f} | Log-reg score var: {log_scores.var():.2f}"
)
print(f"RF score mean: {rf_scores.mean():.2f} | RF score var: {rf_scores.var():.2f}")
print(
    f"XGB score mean: {xgb_scores.mean():.2f} | XGB score var: {xgb_scores.var():.2f}"
)

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)
N_TRIALS = 50

In [ ]:
def objective_log(trial):
    C = trial.suggest_float("C", 1e-3, 1e2, log=True)
    penalty = trial.suggest_categorical("penalty", ["l1", "l2"])

    model = LogisticRegression(
        C=C,
        penalty=penalty,
        solver="liblinear",  # supports both l1 and l2
        class_weight="balanced",
        max_iter=1000,
        random_state=42,
    )

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor_log),
            ("model", model),
        ]
    )

    scores = cross_val_score(pipeline, X, y, cv=cv, scoring="roc_auc", n_jobs=-1)
    return scores.mean()


study_log = optuna.create_study(direction="maximize", study_name="logreg")
study_log.optimize(objective_log, n_trials=N_TRIALS)

print(f"Best log-reg params: {study_log.best_params}")
print(f"Best log-reg ROC-AUC: {study_log.best_value}")

In [ ]:
def objective_rf(trial):
    n_estimators = trial.suggest_int("n_estimators", 100, 800)
    max_depth = trial.suggest_int("max_depth", 2, 20)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
    max_features = trial.suggest_categorical("max_features", ["sqrt", "log2", None])

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        class_weight="balanced",
        random_state=42,
    )

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor_tree),
            ("model", model),
        ]
    )

    scores = cross_val_score(pipeline, X, y, cv=cv, scoring="roc_auc", n_jobs=-1)
    return scores.mean()


study_rf = optuna.create_study(direction="maximize", study_name="random_forest")
study_rf.optimize(objective_rf, n_trials=N_TRIALS)

print(f"Best RF params: {study_rf.best_params}")
print(f"Best RF ROC-AUC: {study_rf.best_value}")

In [ ]:
def objective_xgb(trial):
    n_estimators = trial.suggest_int("n_estimators", 100, 800)
    max_depth = trial.suggest_int("max_depth", 2, 12)
    learning_rate = trial.suggest_float("learning_rate", 1e-3, 0.3, log=True)
    subsample = trial.suggest_float("subsample", 0.5, 1.0)
    colsample_bytree = trial.suggest_float("colsample_bytree", 0.5, 1.0)
    min_child_weight = trial.suggest_int("min_child_weight", 1, 10)
    gamma = trial.suggest_float("gamma", 0, 5)
    reg_alpha = trial.suggest_float("reg_alpha", 1e-3, 10, log=True)
    reg_lambda = trial.suggest_float("reg_lambda", 1e-3, 10, log=True)

    model = XGBClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        min_child_weight=min_child_weight,
        gamma=gamma,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda,
        scale_pos_weight=scale_val_xgb,
        random_state=42,
        eval_metric="logloss",
    )

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor_tree),
            ("model", model),
        ]
    )

    scores = cross_val_score(pipeline, X, y, cv=cv, scoring="roc_auc", n_jobs=-1)
    return scores.mean()


study_xgb = optuna.create_study(direction="maximize", study_name="xgboost")
study_xgb.optimize(objective_xgb, n_trials=N_TRIALS)

print(f"Best XGB params: {study_xgb.best_params}")
print(f"Best XGB ROC-AUC: {study_xgb.best_value}")